In [1]:
from trials_pipeline import fetch_trials, extract_fields, run_pipeline, build_include_exclude_patterns, classify
import pandas as pd

In [2]:
df1 = pd.read_csv("diabetes_2020-01-01_to_2023-12-31_trials.csv", keep_default_na=False)
df1["date_window"] = "2020-01-01 to 2023-12-31"

df2 = pd.read_csv("diabetes_2023-07-01_to_2026-07-01_trials.csv", keep_default_na=False)
df2["date_window"] = "2023-07-01 to 2026-07-01"

df_all = pd.concat([df1, df2], ignore_index=True)
df_all = df_all[df_all["phase"] != "NA"]

In [3]:
lookup = pd.read_csv("industry_sponsor_normalization_USABLE_LOOKUP.csv")
df_all["lead_sponsor_normalized"] = df_all["lead_sponsor"].str.strip().str.lower()
lookup["raw_sponsor_name_normalized"] = lookup["raw_sponsor_name"].str.strip().str.lower()
df_all = df_all.merge(
    lookup[["raw_sponsor_name_normalized", "dashboard_sponsor_name"]],
    left_on="lead_sponsor_normalized",
    right_on="raw_sponsor_name_normalized",
    how="left",
    indicator=True
)
df_all["dashboard_sponsor_name"] = df_all["dashboard_sponsor_name"].fillna(df_all["lead_sponsor"])
df_all["_merge"].value_counts()

_merge
both          962
left_only     464
right_only      0
Name: count, dtype: int64

In [4]:
from trials_pipeline import build_include_exclude_patterns, classify

mesh_dict = pd.read_csv("mesh_include_exclude_dictionary.csv")
mesh_dict = mesh_dict[mesh_dict["Disease Area"] == "diabetes"]

# Include terms use slash-separated synonyms as labels — same expansion pattern as CV and kidney
diabetes_include_keywords = {
    "Type 2 diabetes": ["type 2 diabetes", "type ii diabetes", "t2dm", "type2 diabetes",
                         "diabetes mellitus, type 2", "diabetes mellitus type 2", "diabete type 2"],
    "Type 1 diabetes": ["type 1 diabetes", "type i diabetes", "t1dm", "t1d", "type1 diabetes",
                         "diabetes mellitus, type 1", "diabetes mellitus type 1", "diabete type 1"],
    "Diabetes mellitus unspecified": ["diabetes mellitus", "diabete mellitus", "diabetes (dm)", "diabetes"],
    "Gestational diabetes": ["gestational diabetes"],
    "Prediabetes": ["prediabetes", "pre-diabetes", "pre diabetes"],
    "Rare diabetes subtypes": ["lipoatrophic diabetes", "pancreatogenic diabetes",
                                "maternally inherited diabetes", "midd", "latent autoimmune diabetes", "lada"],
    "Diabetic ketoacidosis": ["diabetic ketoacidosis", "dka"],
}

include_pattern, exclude_pattern = build_include_exclude_patterns(mesh_dict, diabetes_include_keywords)

df_all["indication_classification"] = df_all["conditions"].apply(
    lambda c: classify(c, include_pattern, exclude_pattern, match_label="true_diabetes")
)
df_all["is_drug_biologic"] = df_all["intervention_types"].str.contains("DRUG|BIOLOGICAL", na=False)

In [5]:
review_list = (
    df_all[df_all["indication_classification"] == "needs_review"]["conditions"]
    .str.split("|").explode().value_counts()
)
review_list.to_csv("diabetes_conditions_needing_review.csv")

## Phase Filter

In [6]:
df_phased = df_all[~df_all["phase"].isin(["BLANK", "PHASE2,PHASE3"])]

## Observational split + true-diabetes-drug filter

In [7]:
df_observational = df_all[
    (df_all["study_type"] == "OBSERVATIONAL")
    & (df_all["indication_classification"] == "true_diabetes")
]
df_true_diabetes_drug = df_phased[
    (df_phased["indication_classification"] == "true_diabetes")
    & (df_phased["is_drug_biologic"])
]

## Company and indication views

In [8]:
company_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2"]
company_late_phases = ["PHASE3", "PHASE4"]
indication_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2", "PHASE2"]
indication_late_phases = ["PHASE3", "PHASE4"]

df_company_early = df_true_diabetes_drug[df_true_diabetes_drug["phase"].isin(company_early_phases)]
df_company_late = df_true_diabetes_drug[df_true_diabetes_drug["phase"].isin(company_late_phases)]

top20_companies_all_phases = (
    df_true_diabetes_drug.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_companies_early_phase = (
    df_company_early.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_by_window = (
    df_true_diabetes_drug.groupby(["date_window", "dashboard_sponsor_name"])["nct_id"]
    .nunique().reset_index()
    .sort_values(["date_window", "nct_id"], ascending=[True, False])
    .groupby("date_window").head(20)
)

df_indications = df_true_diabetes_drug.assign(
    conditions=df_true_diabetes_drug["conditions"].str.split("|")
).explode("conditions")

top_indications_early = (
    df_indications[df_indications["phase"].isin(indication_early_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)
top_indications_late = (
    df_indications[df_indications["phase"].isin(indication_late_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)

## Exports

In [9]:
top20_companies_all_phases.to_csv("diabetes_top20_companies_all_phases.csv")
top20_companies_early_phase.to_csv("diabetes_top20_companies_early_phase.csv")
top20_by_window.to_csv("diabetes_top20_companies_by_date_window.csv", index=False)
top_indications_early.to_csv("diabetes_top_indications_early_phase.csv")
top_indications_late.to_csv("diabetes_top_indications_late_phase.csv")
df_observational.to_csv("diabetes_observational_studies.csv", index=False)

## Tableau Prep

In [10]:
df_all.to_csv("diabetes_full_dataset_for_tableau.csv", index=False)

## Collaborators explode

In [11]:
df_collab = df_all.assign(
    collaborators=df_all["collaborators"].str.split("|")
).explode("collaborators")
df_collab = df_collab[df_collab["collaborators"].notna() & (df_collab["collaborators"] != "")]
df_collab.to_csv("diabetes_collaborations.csv", index=False)